## Graph Intelligence: Google ADK + Neo4j MCP
This notebook demonstrates how to build a generative AI agent that can autonomously interact with a Neo4j Graph Database using the Model Context Protocol (MCP).

### 1. Environment Setup & Configuration
First, we install the necessary libraries.

• google-adk: The framework for building agents.  
• neo4j: The driver for database connectivity.  
• mcp: The protocol layer for tool communication.

In [ ]:
# Install core dependencies
%pip install -q google-adk neo4j mcp requests neo4j-mcp-server

#### Imports

In [2]:
import os
import base64
import json
import subprocess
import time
import requests
import warnings
from getpass import getpass
from neo4j import GraphDatabase
from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams
from google.adk.tools.function_tool import FunctionTool
import neo4j_mcp_server

warnings.filterwarnings("ignore", category=UserWarning)

#### LLM Configuration

In [3]:
os.environ["GOOGLE_API_KEY"] = getpass("Enter Google API Key: ")
os.environ["GEMINI_MODEL"] = "gemini-flash-latest"

#### DB configuration

In [4]:
os.environ["NEO4J_URI"] = "neo4j+s://demo.neo4jlabs.com"
os.environ["NEO4J_DATABASE"] = "companies"

### 2. Toolset Configuration
We will connect the ADK to the MCP server via http method.

This starts a persistent background process.  
**Note:** Authentication is handled via a Base64 encoded header rather than environment variables in the transport layer. Refer official [documentation](https://neo4j.com/docs/mcp/current/installation/)

In [5]:
# Encode credentials for HTTP Header
credentials = base64.b64encode(b"companies:companies").decode()
PORT = "8443"

# Start background server
process = subprocess.Popen(
    [
        "neo4j-mcp-server",
        "--neo4j-uri", os.environ["NEO4J_URI"],
        "--neo4j-transport-mode", "http",
        "--neo4j-http-port", PORT,
        "--neo4j-database", os.environ["NEO4J_DATABASE"],
    ],
    stdout=subprocess.PIPE, 
    stderr=subprocess.PIPE, 
    text=True
)

time.sleep(3) # Warm-up time

# Check if the process started
poll = process.poll()
if poll is not None:
    stdout, stderr = process.communicate()
    print(f"Server failed to start (Exit Code: {poll})")
    print(f"STDOUT: {stdout}")
    print(f"STDERR: {stderr}")
else:
    print(f"Server started successfully on port {PORT}")


Server started successfully on port 8443


In [6]:
#Defining the tool with HTTP connection parameters, including the Authorization header
mcp_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"http://localhost:{PORT}/mcp",
        headers={"Authorization": f"Basic {credentials}"}
        ),
)

### 3. Agent Construction
We wrap the ADK LlmAgent creation in an async function. The agent is provided with the mcp_tools we defined above.

In [7]:
async def create_agent(model, name, prompt, tools_list):
    """
    Constructs an LLM Agent with specialized tools.
    The agent uses the system prompt to understand it should favor 'get-schema' 
    before attempting complex queries.
    """
    return LlmAgent(
        model=model,
        name=name,
        instruction=prompt,
        tools=tools_list
    )

system_prompt = """
You are a graph database assistant. Your job is to answer user questions by querying Neo4j.

Always run 'get-schema' first if you are unfamiliar with the graph structure.
Use Cypher queries to retrieve data.
If the data is not found, state that clearly.
"""

mcp_agent = await create_agent(
    model=os.environ["GEMINI_MODEL"], 
    name="neo4j_explorer", 
    prompt=system_prompt, 
    tools_list=[mcp_tools]
)

### 4. Execution Engine
The Runner orchestrates the conversation, handling tool calls (the "back-and-forth" between Gemini and Neo4j) automatically.

In [8]:
async def ask_graph(agent, query):
    session_service = InMemorySessionService()
    artifacts_service = InMemoryArtifactService()

    session = await session_service.create_session(state={}, app_name="neo4j_app", user_id="user_1")
    content = types.Content(role='user', parts=[types.Part(text=query)])

    runner = Runner(
        app_name="neo4j_app",
        agent=agent,
        artifact_service=artifacts_service,
        session_service=session_service,
    )

    print(f"Processing: {query}")
    try:
        events_async = runner.run_async(
            session_id=session.id, 
            user_id=session.user_id, 
            new_message=content
        )

        async for event in events_async:
            if hasattr(event, 'content') and event.content:
                for part in event.content.parts:
                    if part.text:
                        print(f"\nResult: {part.text}")
    except Exception as e:
        print(f"Error: {e}")

# Example Query
await ask_graph(mcp_agent, "How many people are in the database?")

Processing: How many people are in the database?



Result: There are 8,064 people in the database.


### 5. Custom tools
Beyond using existing MCP servers, you can also implement your own custom tools and add them directly to the agent. This allows you to create specialized functionality tailored to your specific use case. Custom tools can be implemented using the @tool decorator, which turns any function into a tool the agent can invoke.

Here, we use GraphDatabase from the neo4j package to establish a connection to our database and build a tool that queries investment relationships, giving you more control over the query logic.

In [9]:
async def get_investments(company: str) -> str:
    """
    Returns the investments by a company by name. 
    Returns list of investment ids, names and types.
    """
    query = """
    MATCH (o:Organization)-[:HAS_INVESTOR]->(i)
    WHERE o.name = $company
    RETURN i.id as id, i.name as name, head(labels(i)) as type
    """
    try:
        driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]), database=os.environ["NEO4J_DATABASE"])
        records, _, _ = driver.execute_query(
            query, 
            company=company
        )
        results = [record.data() for record in records]
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"Error fetching investments: {str(e)}"


Transforming a Python function into a tool is a straightforward way to integrate custom logic into your agents. When you assign a function to an agent’s tools list, the framework automatically wraps it as a FunctionTool.

In [10]:
custom_investment_tool = FunctionTool(get_investments)

In [11]:
custom_tools = [mcp_tools, custom_investment_tool]
custom_agent_prompt = "You are a helpful assistant with access to a Neo4j graph database containing company data. Use the available tools to query the database and answer questions."
custom_agent_name = "Custom_neo4j_agent"
custom_agent = await create_agent(os.environ["GEMINI_MODEL"], custom_agent_name, custom_agent_prompt, custom_tools)

In [12]:
query = "Which companies did Google invest in?"
await ask_graph(custom_agent, query)

Processing: Which companies did Google invest in?

Result: Google (including its parent company Alphabet and investment arms like GV and CapitalG) has invested in a wide range of companies across various sectors. Based on the database, here are some of the companies Google has invested in:

### Venture Investments (GV, CapitalG, and Direct)
Google's investment arms have backed numerous high-profile tech companies:
*   **GV (formerly Google Ventures):** Neo4j, Slack, DocuSign, Plaid, Vercel, Cockroach Labs, Synthesia, Duo Security, Recorded Future, Shape Security, Intercom, Cohesity, Cloudera, and Segment.
*   **CapitalG:** Databricks, Dataiku, Cloudflare, Zscaler, Looker, Glassdoor, and Momentive.
*   **Direct Google Investments:** Cloudflare, Trifacta, Ionic Security, Avere Systems, and FlexiDAO.
*   **Other Arms:** Wysa and Botsociety (via Google Assistant Investments); Papaya Global and GrowthSpace (via Google for Startups).

### Major Acquisitions (Subsidiaries)
In addition to vent

### 7. Google adk plugin integration
  In ADK, Plugins are powerful interceptors that allow you to inject cross-cutting concerns—like long-term memory, observability, or safety guardrails—without cluttering your agent's core logic.  
  Here, we demonstrate how to wrap the neo4j-agent-memory capability into a reusable ADK Plugin. This effectively "upgrades" any standard agent into a stateful, learning assistant simply by registering the plugin at runtime.  
We will implement a "Contextual Analyst" that remembers a user's investment focus and prioritizes those insights when exploring the company graph.

We will use Neo4j's [agent memory](https://github.com/neo4j-labs/agent-memory) - A graph-native memory system for AI agents.

In [ ]:
%pip install -q neo4j-agent-memory[google-adk,openai]

In [11]:
from pydantic import SecretStr
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.google_adk import Neo4jMemoryService
from neo4j_agent_memory.config.settings import Neo4jConfig, EmbeddingConfig, EmbeddingProvider, ExtractionConfig, ExtractorType

In [ ]:
# Use separate credentials for the Memory Graph to keep it distinct from the Business Data
os.environ["MEMORY_NEO4J_URI"] = "" #Enter your Neo4j URI here
os.environ["MEMORY_NEO4J_USERNAME"] = "" #Enter your Neo4j Username here
os.environ["MEMORY_NEO4J_PASSWORD"] = "" #Enter your Neo4j Password here

In [13]:
os.environ["OPENAI_API_KEY"]  = getpass("Enter OPENAI API Key: ")

##### Setup the Memory Client
we will use vertex ai embedding for storing memory embeddings. You can refer other embedding configurations [here](https://github.com/neo4j-labs/agent-memory/blob/main/README.md#programmatic-configuration)

In [27]:

mem_settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=os.environ["MEMORY_NEO4J_URI"],
        username=os.environ["MEMORY_NEO4J_USERNAME"],
        password=SecretStr(os.environ["MEMORY_NEO4J_PASSWORD"]),
        database="neo4j",
    ),
    embedding=EmbeddingConfig(
        provider=EmbeddingProvider.OPENAI,
        model="text-embedding-ada-002",
    ),
    extraction=ExtractionConfig(
        extractor_type=ExtractorType.PIPELINE,
        enable_spacy=False,   
        enable_gliner=False,  
        enable_llm_fallback=True,
        llm_model="gpt-4o-mini"
    )
)

mem_client = MemoryClient(mem_settings)
await mem_client.__aenter__()


memory_service = Neo4jMemoryService(
    memory_client=mem_client,
    user_id="analyst_01",
    include_entities=True,
    include_preferences=True
)

We will configure new agent memoryAgent

In [28]:
memory_agent_prompt = """
You are a helpful assistant with access to a Neo4j graph database AND a long-term memory of the user's details.

YOUR PRIORITY:

First, check the "CONTEXT FROM PREVIOUS CHATS" provided in the prompt. If the answer is there (e.g., user's name, preferences), USE IT directly. Do not call tools.
If the answer is not in the context, use the available tools to query the Neo4j database.
If the data is not found in either, state that clearly.


Always answer politely.
"""
memory_agent_name = "neo4j_analyst_agent"
memory_agent = await create_agent(os.environ["GEMINI_MODEL"], memory_agent_name, memory_agent_prompt, custom_tools)

We will now implement a custom BasePlugin. This plugin will intercept the conversation loop at two specific lifecycle events:
1. on_user_message: Before the LLM sees the user's query, we check our Graph Memory for relevant context and inject it.
2. after_model: After the LLM responds, we capture the interaction and save it back to the Graph Memory for future recall.

In [29]:
from google.adk.plugins.base_plugin import BasePlugin

class MarketAnalystPlugin(BasePlugin):
    def __init__(self, memory_service):
        super().__init__(name="neo4j_analyst_app")
        self.memory_service = memory_service
        self.current_user_query = None
        self.current_session_id = None

    async def on_user_message_callback(self, *, invocation_context, user_message: types.Content):
        """
        Triggered when the user sends a message. 
        We search for context and inject it into the prompt.
        """
        self.current_session_id = invocation_context.session.id

        # Safety check: Ensure text exists
        if not user_message.parts or not user_message.parts[0].text:
            return

        user_text = user_message.parts[0].text
        self.current_user_query = user_text # Store for saving later

        try:
            memories = await self.memory_service.search_memories(query=user_text, limit=5, threshold=0.2)
        except Exception as e:
            print(f" [Plugin] Search skipped: {e}")
            memories = []

        if memories:
            print(f"   ↳ Found {len(memories)} memories.")
            for i, r in enumerate(memories):
                print(f"   ↳ Memory {i+1}: {r.content[:100]}...") # Print first 100 chars of each memory
            context_str = "\n".join([f"- {r.content}" for r in memories])

            # ADK Context Injection
            user_message.parts[0].text = (
                f"User Question: {user_text}\n\n"
                f"--- MEMORY CONTEXT ---\n"
                f"{context_str}\n"
                f"----------------------\n"
                f"Instructions: Use the MEMORY CONTEXT to answer if relevant. "
                f"Otherwise, use your tools."
            )
        return None

    async def after_model_callback(self, *, callback_context, llm_response):
        """
        Triggered after the model responds.
        We only save the response if it is actual TEXT (ignoring tool calls).
        """
        if not llm_response.content or not llm_response.content.parts:
            return None

        part = llm_response.content.parts[0]

        if hasattr(part, 'text') and part.text and self.current_user_query:
            agent_text = part.text

            print(f" [Plugin] Saving interaction...")
            try:
                await self.memory_service.add_session_to_memory({
                    "id": self.current_session_id,
                    "messages": [
                        {"role": "user", "content": self.current_user_query},
                        {"role": "assistant", "content": agent_text}
                    ]
                })
                print("   ↳ Saved.")
            except Exception as e:
                print(f" [Plugin] Save failed: {e}")

        return None

High-Value Execution Workflow
We will now run a two-part scenario:
1. Define Strategy: Tell the agent our specific focus.
2. Contextual Query: Ask a broad question and see how it uses "Memory" to provide a specialized answer.

In [30]:
async def run_analyst_task(agent, query):
    plugin = MarketAnalystPlugin(memory_service)
    session_service = InMemorySessionService()
    artifacts_service = InMemoryArtifactService()
    runner = Runner(
        app_name="neo4j_analyst_app",
        agent=agent,
        session_service=session_service,
        artifact_service=artifacts_service,
        plugins=[plugin] 
    )
    session = await session_service.create_session(state={}, app_name="neo4j_analyst_app", user_id="analyst_01")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    print(f"\n[USER]: {query}")
    
    async for event in runner.run_async(session_id=session.id, user_id="analyst_01", new_message=content):
        if hasattr(event, 'content') and event.content:
            for part in event.content.parts:
                if part.text:
                    print(f"[AGENT]: {part.text}")

# STEP 1: Querying with Memory Context Injection
Query1 = "I am conducting a competitive analysis of 'Google'. I am specifically worried about their subsidiaries and who their top-tier competitors are in the AI space."
await run_analyst_task(memory_agent, Query1)

print("\n--- System Indexing Memory... ---\n")
import asyncio
await asyncio.sleep(5) # Give the vector index a moment

# STEP 2: Recalling from Memory and Neo4j
Query2 = "What are the main risks in the supply chain for the company I am currently tracking?"
await run_analyst_task(memory_agent, Query2)


[USER]: I am conducting a competitive analysis of 'Google'. I am specifically worried about their subsidiaries and who their top-tier competitors are in the AI space.
 [Plugin] Saving interaction...
   ↳ Saved.
[AGENT]: Google is a dominant force in the AI landscape, and its competitive analysis reveals a robust ecosystem of subsidiaries and high-stakes competition.

### **Google’s Key AI Subsidiaries**
While Google (under Alphabet Inc.) has hundreds of subsidiaries, several are core to its AI strategy:

*   **DeepMind Technologies:** Acquired in 2014, DeepMind is a premier AI research lab responsible for breakthroughs like AlphaGo and AlphaFold. It is now deeply integrated with Google’s primary AI teams.
*   **Google AI & Google Research:** These divisions focus on fundamental machine learning research, developing models like LaMDA, PaLM, and Gemini.
*   **Google Cloud:** This subsidiary offers the **Vertex AI** platform, competing directly in the enterprise AI and infrastructure mar

# Summary

This notebook demonstrates how to use Google's Agent Development Kit (ADK) to build conversational agents that interact with a Neo4j graph database. The workflow includes:

1. Agent Construction: We built a gemini-flash agent and equipped it with MCP tools for database access.
2. Tool Abstraction: We integrated standard neo4j-mcp tools alongside custom Python functions (get_investments), showing how ADK handles diverse tool types uniformly.
3. Plugin Architecture: Instead of hardcoding memory logic, we utilized the BasePlugin class. This allowed us to drop in neo4j-agent-memory as a modular component.  
• The plugin intercepted the user input to inject context (RAG).  
• It observed the output to save state (Memory).  
  
This architecture ensures your agent logic remains clean while capabilities like Memory, Guardrails, or Logging can be added or removed as plugins.